# 03 · Model training

| Stage | Notebook |
|---|---|
| 1. Data quality & EDA | `01_eda.ipynb` |
| 2. Feature engineering | `02_feature_engineering.ipynb` |
| **3. Model training** | **this notebook** |
| 4. Evaluation | `04_evaluation.ipynb` |

This notebook recomputes training using the same `aerocast.pipeline` functions as `scripts/run_phase1.py`. Its computed tables are checked against saved `results/` artefacts. These assertions do not check handwritten explanations. Runtime depends on the machine and experiment.

Model configuration/family selection uses training-window development CV. A separate tree-count diagnostic also uses Sep–Oct, and the split summary displays the test cohort's size/base rate. Previous development-time test exposure is disclosed in notebook 04, §0.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np, pandas as pd
from IPython.display import display
from aerocast import artefacts, config, models, pipeline, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 160)
R = config.RESULTS_DIR
S = json.loads((R / "summary.json").read_text())

## 1. Chronological split

In [2]:
P = pipeline.prepare()
splits = pd.DataFrame(P.split_summary()).T
pd.testing.assert_frame_equal(splits, pd.DataFrame(S["splits"]).T)   # == summary.json
splits.assign(role=["fitting + rolling-origin CV", "calibration + threshold choice", "exploratory held-out evaluation"])

,rows,delay_rate,from,to,role
train,211684,0.2649,2013-01-08,2013-08-31,fitting + rolling-origin CV
calib,55628,0.157,2013-09-01,2013-10-31,calibration + threshold choice
test,53991,0.2583,2013-11-01,2013-12-31,exploratory held-out evaluation


* **Jan 1–7 excluded** as history warm-up.
* The table counts scoring rows. Final classifiers fit **211,517 eligible training labels**, excluding **167** whose outcomes were unavailable at the model deadline. CV has its own availability masks.
* **Base rates:** 26.5% training, 15.7% calibration, 25.8% test; November 18.1%, December 33.5%.
* The test period was reserved for evaluation but seen during development. Its results are exploratory, as disclosed in notebook 04.

## 2. The two models: theory in brief

**Logistic regression.** It models $P(y=1\mid x) = \sigma(w^\top x + b)$ and minimises the **L2-regularised log-loss**

$$\mathcal{L}(w) = -\frac1n\sum_i \big[y_i\log p_i + (1-y_i)\log(1-p_i)\big] + \frac{1}{2Cn}\lVert w\rVert^2 .$$

The penalty shrinks coefficients toward 0. That's needed here because the history features are strongly correlated (notebook 01, §8). Smaller *C* means stronger shrinkage. The objective is convex; convexity alone does not establish a unique optimum without additional conditions.

**XGBoost.** An additive ensemble of trees $F_T(x)=\sum_t f_t(x)$, fitted stage-wise. At stage *t* it minimises a **second-order Taylor approximation** of the loss plus a complexity penalty:

$$\tilde{\mathcal L}^{(t)} = \sum_i \big[g_i f_t(x_i) + \tfrac12 h_i f_t(x_i)^2\big] + \gamma T_{\text{leaves}} + \tfrac12\lambda\lVert w\rVert^2,$$

with $g_i, h_i$ the first and second derivatives of the log-loss at the current prediction ($g_i = p_i - y_i$, $h_i = p_i(1-p_i)$). For a fixed tree structure, the optimal leaf weight is $w_j^* = -G_j/(H_j+\lambda)$, where $G_j, H_j$ are the leaf's sums of gradients and Hessians. The split gain uses the same quantities. Regularisation comes from λ (`reg_lambda`), `min_child_weight` (minimum Hessian mass per leaf), depth, shrinkage (`learning_rate`), row/column subsampling, and the number of trees.

**Why log-loss?** It's a **strictly proper scoring rule**: its expected value is minimised only by reporting the true probability. A model trained on log-loss is therefore pushed toward calibrated probabilities on its training distribution. (The Brier score is also proper. Accuracy and AUC are not; AUC only measures ranking.)

## 3. Baselines (reference rows next to every learned model)

* **Prior (base rate)**: the training base rate for every flight (PR-AUC = the evaluated window's base rate, ROC-AUC = 0.5);
* **Route history**: the smoothed 14-day carrier-route delay rate on its own, no model.

In [3]:
raw_preds = pipeline.baseline_preds(P)
base_scores = pipeline.mean_scores(raw_preds)
print(artefacts.assert_matches(base_scores, "mean_scores.csv", key="model"))
base_scores.set_index("model").add_prefix("mean score · ")

OK: identical to results/mean_scores.csv


,mean score · train,mean score · calib,mean score · test
model,,,
prior,0.265,0.2650,0.265
route_history,0.265,0.1574,0.242


## 4. Why rolling-origin CV instead of one validation month

The first design tuned on **September** alone. September is the calmest month of the year (13.7% late), and its scores didn't track later months. The diagnostic below fits XGBoost (final hyperparameters) on Jan–Aug and plots PR-AUC on September and on October as trees are added. It uses the calibration window only, never test.

In [4]:
curve = pipeline.single_month_tree_curve(P, S["xgb_params"])
print(artefacts.assert_matches(curve, "single_month_tree_curve.csv"))
fig = plots.tree_count_curve(curve, S["xgb_params"]["n_estimators"])
plots.save(fig, "train_single_month_tree_curve"); fig

OK: identical to results/single_month_tree_curve.csv


<Figure size 650x320 with 1 Axes>

In [5]:
pd.DataFrame({m: {"best n_trees": int(curve.loc[curve[c].idxmax(), "n_trees"]), "best PR-AUC": curve[c].max(),
                   "PR-AUC range": curve[c].max() - curve[c].min()} for m, c in [("Sep", "pr_auc_m9"), ("Oct", "pr_auc_m10")]}).T

,best n_trees,best PR-AUC,PR-AUC range
Sep,80.0,0.2418,0.0277
Oct,200.0,0.3073,0.0384


Two adjacent months give different "best" tree counts, on nearly flat curves, so a single month is a noisy selector. **Rolling-origin CV** inside the training window replaces it:

| Fold | Train | Validate |
|---|---|---|
| 1 | Jan–Apr | May |
| 2 | Jan–May | Jun |
| 3 | Jan–Jun | Jul |
| 4 | Jan–Jul | Aug |

Training never sees a month at or after the one it's validated on (asserted in `tests/`).

## 5. Logistic regression: C by rolling CV

In [6]:
lr = pipeline.train_logreg(P)
print(artefacts.assert_matches(lr.cv.reset_index(drop=True), "cv_logreg.csv"))
wide = lr.cv.pivot(index="C", columns="fold", values="val_pr_auc")
wide.assign(mean=wide.mean(axis=1), std=wide.std(axis=1))

OK: identical to results/cv_logreg.csv


fold,0,1,2,3,mean,std
C,,,,,,
0.0003,0.4257,0.5011,0.5444,0.3944,0.4664,0.0686
0.0010,0.4299,0.5033,0.5483,0.3944,0.4690,0.0697
0.0030,0.4334,0.5044,0.5505,0.3942,0.4706,0.0701
0.0100,0.4347,0.5051,0.5515,0.3938,0.4713,0.0705
0.1000,0.4350,0.5055,0.5523,0.3933,0.4715,0.0710
1.0000,0.4344,0.5050,0.5521,0.3932,0.4712,0.0710


In [7]:
print("chosen C =", lr.C)
fig = plots.cv_logreg_curve(lr.cv); plots.save(fig, "train_cv_logreg"); fig

chosen C = 0.1


<Figure size 600x320 with 1 Axes>

The curve is flat above C ≈ 0.003. C = 0.1 is the (marginal) maximum and lies inside the grid. The fold-to-fold spread (Aug ≈ 0.39 vs Jul ≈ 0.55) is far larger than any difference between C values, and it tracks each month's base rate: PR-AUC rises with the share of late flights.

*Joint tuning with the history settings.* The smoothing strength m and the long window were first scored at a fixed C = 0.001. Notebook 02, §6 re-tunes (m, window) **jointly with C** (each pair scored at its own best C): the same pair wins and C = 0.1 again, so the earlier choice stands.

## 6. XGBoost: configuration search

**Grid** (`models.XGB_GRID`, 48 combinations, 16 sampled at random):
* depth {2, 3, 5}: the order of interactions;
* min_child_weight {20, 100}: minimum Hessian mass per leaf, which matters for a weak, noisy signal;
* colsample {0.5, 0.8} and λ {1, 10}: decorrelation and leaf shrinkage;
* learning rate {0.03, 0.1}: shrinkage, traded off against the number of trees, which early stopping sets.

For each configuration and fold: up to 3,000 trees with **early stopping** (100 rounds) on that fold's validation month. Score = mean fold PR-AUC; tree count = mean of the four best iterations.

In [8]:
xgb = pipeline.train_xgb(P, n_configs=16)
print(artefacts.assert_matches(xgb.cv_configs.reset_index(), "cv_xgb_configs.csv"))
xgb.cv_configs.head(8)

OK: identical to results/cv_xgb_configs.csv


,mean_pr_auc,std_pr_auc,n_trees
config,,,
15,0.4761,0.0703,148
0,0.4728,0.0688,98
5,0.4709,0.0665,204
1,0.4693,0.0692,373
11,0.4680,0.0674,337
14,0.4675,0.0671,104
10,0.4671,0.0665,224
4,0.4623,0.0625,63


In [9]:
assert xgb.params == S["xgb_params"]
pd.Series({**xgb.params, "cv_mean_pr_auc (early-stopped)": xgb.cv_configs.iloc[0]["mean_pr_auc"],
           "cv_std_pr_auc": xgb.cv_configs.iloc[0]["std_pr_auc"]}, name="chosen").to_frame()

,chosen
max_depth,2.0000
learning_rate,0.1000
min_child_weight,20.0000
subsample,0.8000
colsample_bytree,0.5000
reg_lambda,1.0000
n_estimators,148.0000
cv_mean_pr_auc (early-stopped),0.4761
cv_std_pr_auc,0.0703


The selected configuration uses **depth-2 trees**, learning rate 0.1, min_child_weight **20**, colsample **0.5**, λ **1.0**, subsample 0.8, and **148 trees**. Its in-fold early-stopping search score is **0.4761 ± 0.0703**. The fixed-tree family-comparison score is **0.4474 ± 0.0686**.

Both the configuration and the fold-averaged tree count come from these development folds; the resulting CV scores are not independent nested-CV estimates.

## 7. Which model is final? (development CV rule)

**Rule:** select the model family with higher mean rolling-CV AP, using XGBoost's fixed, fold-averaged tree count for the comparison.

Early stopping selects a tree count on each validation fold, making XGBoost's search score optimistic. The fixed-tree comparison removes that particular in-fold stopping advantage. It still uses development folds involved in selecting the configuration and tree count. Logistic Regression's C is also selected on these folds. Neither family's best reported CV score is an independent estimate after nested tuning.

In [10]:
cv = pipeline.cv_summary(lr, xgb, P)
cv_tbl = pipeline.cv_table(cv)
print(artefacts.assert_matches(cv_tbl, "cv_summary.csv"))
cv_tbl.drop(columns="variant")

OK: identical to results/cv_summary.csv


,model,calibration,cv_pr_auc_mean,cv_pr_auc_std,fold_m5,fold_m6,fold_m7,fold_m8
0,Prior (base rate),none,0.2777,0.0481,0.2267,0.3152,0.3221,0.2467
1,Route history,none,0.3477,0.0601,0.2766,0.3911,0.4034,0.3197
2,LogReg (as deployed),raw,0.4715,0.0710,0.4350,0.5055,0.5523,0.3933
3,"XGBoost (as deployed, fixed trees)",raw,0.4474,0.0686,0.4338,0.4193,0.5466,0.3899
4,"XGBoost (in-fold early stopping, optimistic; n...",raw,0.4761,0.0703,0.4400,0.5054,0.5590,0.3999


In [11]:
final = pipeline.select_final_model(cv)
print("final model:", final, "| config.FINAL_MODEL:", config.FINAL_MODEL)
pair = pipeline.paired_cv_difference(cv)
assert pair == S["cv_paired_logreg_minus_xgboost"]
pd.Series({"per-fold LogReg − XGBoost (May..Aug)": np.round(pair["diffs"], 4).tolist(),
           "mean difference": pair["mean"], "sd of difference": pair["std"],
           "folds LogReg better": f'{pair["folds_a_better"]}/{pair["n_folds"]}'}, name="paired").to_frame()

final model: logreg | config.FINAL_MODEL: logreg


,paired
per-fold LogReg − XGBoost (May..Aug),"[0.0012, 0.0862, 0.0057, 0.0034]"
mean difference,0.0241
sd of difference,0.0414
folds LogReg better,4/4


### What the evidence supports

* Fixed-tree development CV AP: **LogReg 0.4715 versus XGBoost 0.4474**. LogReg leads in all four folds; paired differences are +0.0012, +0.0862, +0.0057, +0.0034 (mean +0.0241, sd 0.0414). Four folds do not establish population-level superiority.
* The optimistic XGBoost in-fold search score is **0.4761**; using it would reverse the selection.
* LogReg has the higher development CV mean (0.4715 versus 0.4474), so it is selected by the declared rule. The test AP-difference interval (XGBoost − LogReg, [-0.013, +0.037]) includes zero: it does not establish a test-performance difference, equivalence, or noninferiority. Test exposure during development remains disclosed.
* These results do not establish that the predictive signal is mostly additive or that useful interactions have been exhausted. An embedding MLP remains a later-phase hypothesis to test fairly.

## 8. Final models (fit on eligible Jan–Aug labels)

In [12]:
raw_preds["logreg"] = pipeline.predict_splits(lr.model, P)
raw_preds["xgboost"] = pipeline.predict_splits(xgb.model, P)
scores = pipeline.mean_scores(raw_preds)
print(artefacts.assert_matches(scores, "mean_scores.csv"))
scores.set_index("model").add_prefix("mean score · ")

OK: identical to results/mean_scores.csv


,mean score · train,mean score · calib,mean score · test
model,,,
prior,0.265,0.2650,0.2650
route_history,0.265,0.1574,0.2420
logreg,0.265,0.2304,0.2584
xgboost,0.265,0.1951,0.2322


The raw mean Sep–Oct scores are **0.2304 for LogReg** and **0.1951 for XGBoost**, above the observed 15.7% delay rate. That establishes aggregate overprediction in this period, without isolating its cause. Notebook 04 applies the declared calibration-selection rule.

## 9. Consistency with `scripts/run_phase1.py`

In [13]:
assert lr.C == S["logreg_C"] and xgb.params == S["xgb_params"]
assert final == S["final_model"]["model"]
assert all(np.allclose(cv[m]["folds"], S["cv_pr_auc"][m]["folds"]) for m in cv)

_, saved_raw = pipeline.load_predictions()
for m in pipeline.RAW_SCORES:
    for s in config.SPLITS:
        assert np.array_equal(raw_preds[m][s].astype(saved_raw[m][s].dtype), saved_raw[m][s]), (m, s)
print("OK: every table above, the hyper-parameters, the final-model choice and all predictions match results/")

OK: every table above, the hyper-parameters, the final-model choice and all predictions match results/
